# 27. Production Monitoring

**Tier:** Evaluation & Production
**Estimated time:** 55 minutes
**Prerequisites:** 21, 24, 26
**Priority:** 🟡 Important — shipping without observability is flying blind, but the tooling itself (LangSmith and similar) is learnable in days and often owned by a platform team. *If skipped, revisit when:* the week before your first production deploy, or the first time a stakeholder asks "why did quality drop last Tuesday?"
**Source material:** Stanford Lecture 8 + @sairahul1 AI Engineer Roadmap — https://x.com/sairahul1/status/2062809249064141017

## What You'll Learn
- Tracing a live system with LangSmith (reusing the `@traceable` pattern from notebook 21)
- Cost and latency tracking as first-class metrics, not afterthoughts
- Detecting quality drift on a simulated traffic log using the eval harness from notebook 24
- Canary / A-B rollout: measuring whether a prompt change actually helped before it ships to everyone

## Why This Matters
An eval harness (notebook 24) tells you if a *version* is good before you ship it. Production monitoring tells you if the *live system* is still good after you ship it — traffic patterns shift, providers have incidents, prompts drift as people edit them. Without this, the first sign of a regression is a user complaint, not a dashboard.


In [ ]:
import os, time, json
import numpy as np
import matplotlib.pyplot as plt

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
HAS_LANGSMITH = bool(os.environ.get("LANGSMITH_API_KEY"))
TEACH_MODEL = "claude-haiku-4-5-20251001"

if HAS_ANTHROPIC:
    import anthropic
    client = anthropic.Anthropic()
    print("Anthropic ready.")
else:
    client = None
    print("No ANTHROPIC_API_KEY — live-call cells will be skipped.")

def ask(prompt, system="You are concise.", max_tokens=200, temperature=0.7):
    if not HAS_ANTHROPIC:
        return "[skipped: no ANTHROPIC_API_KEY]", 0.0
    t0 = time.time()
    try:
        msg = client.messages.create(model=TEACH_MODEL, max_tokens=max_tokens, system=system,
                                      temperature=temperature, messages=[{"role": "user", "content": prompt}])
        latency = time.time() - t0
        return msg.content[0].text, latency
    except Exception as e:
        return f"[skipped: {type(e).__name__}: {str(e)[:120]}]", time.time() - t0


## Tracing with LangSmith

Notebook 21 introduced `@traceable` for wrapping harness calls so they show up in the LangSmith dashboard. Production monitoring is the same primitive, applied continuously: every real request gets traced, so when something goes wrong you have the actual inputs/outputs/latency, not just a vague bug report.

In [ ]:
if HAS_LANGSMITH:
    os.environ.setdefault("LANGSMITH_TRACING", "true")
    os.environ.setdefault("LANGSMITH_PROJECT", os.environ.get("LANGSMITH_PROJECT", "ai-learning-notebooks"))
    try:
        from langsmith import traceable

        @traceable(run_type="chain", name="production_request")
        def handle_request(user_query):
            answer, latency = ask(user_query, max_tokens=100)
            return {"answer": answer, "latency_s": latency}

        result = handle_request("Summarize what a vector database is in one sentence.")
        print("Traced request complete (view in your LangSmith dashboard).")
        print(result)
    except Exception as e:
        print(f"LangSmith tracing skipped (non-fatal): {type(e).__name__}: {str(e)[:120]}")
else:
    print("No LANGSMITH_API_KEY — tracing skipped. Set it in .env to record runs to the LangSmith UI.")


## Cost and latency as first-class metrics

A request that's "correct" but takes 8 seconds or costs 50x the budget is still a production problem. Track cost and latency alongside quality from day one — the shared `ask()` helper above already returns latency; token counts convert directly to cost using the model's per-token price.

In [ ]:
# Approximate Claude Haiku pricing (USD per million tokens) — check current pricing before using in prod.
PRICE_PER_M_INPUT = 1.00
PRICE_PER_M_OUTPUT = 5.00

def estimate_cost(input_tokens, output_tokens):
    return (input_tokens / 1e6) * PRICE_PER_M_INPUT + (output_tokens / 1e6) * PRICE_PER_M_OUTPUT

queries = [
    "What is a vector database?",
    "Explain retrieval-augmented generation in one sentence.",
    "What does KV caching do?",
]

records = []
for q in queries:
    answer, latency = ask(q, max_tokens=80)
    # Rough token estimate for teaching purposes: ~4 chars/token.
    in_tok, out_tok = len(q) // 4, len(answer) // 4
    records.append({
        "query": q, "latency_s": round(latency, 3),
        "cost_usd": round(estimate_cost(in_tok, out_tok), 6),
    })

for r in records:
    print(r)
print(f"\nTotal cost: ${sum(r['cost_usd'] for r in records):.6f}  |  "
      f"Avg latency: {np.mean([r['latency_s'] for r in records]):.2f}s")


## Simulating a traffic log with drift

Real production traffic isn't static — the underlying task can get subtly harder over time (new edge cases, adversarial inputs, model updates upstream). We'll simulate a day of traffic where quality silently degrades partway through, then use the notebook-24 eval harness to detect it — this is the same shape of problem as the memorization demo in notebook 25, but happening live instead of at training time.

In [ ]:
np.random.seed(1)

def normalize(s):
    import re
    return re.sub(r"[^a-z0-9]", "", s.lower())

def exact_match_score(output, reference):
    return 1.0 if normalize(reference) in normalize(output) else 0.0

# Simulate 40 requests across a "day": quality (probability of a correct response)
# silently drops at request #25, as if an upstream prompt change or model swap regressed something.
N_REQUESTS = 40
DRIFT_POINT = 25
GOOD_QUALITY, BAD_QUALITY = 0.95, 0.55

simulated_scores = []
for i in range(N_REQUESTS):
    p_correct = GOOD_QUALITY if i < DRIFT_POINT else BAD_QUALITY
    simulated_scores.append(1.0 if np.random.random() < p_correct else 0.0)

print(f"Overall accuracy across the day: {np.mean(simulated_scores):.0%} (this single number hides the drift)")


In [ ]:
%matplotlib inline
window = 8
rolling = [np.mean(simulated_scores[max(0, i - window):i + 1]) for i in range(len(simulated_scores))]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(rolling, marker="o", markersize=3)
ax.axvline(DRIFT_POINT, color="red", linestyle="--", label="actual regression point")
ax.axhline(0.8, color="gray", linestyle=":", label="alert threshold")
ax.set_xlabel("request #")
ax.set_ylabel(f"rolling accuracy (window={window})")
ax.set_title("Drift detection: rolling accuracy catches what the daily average hides")
ax.legend()
plt.tight_layout()
plt.show()


*The overall daily average looks acceptable, but the rolling-window accuracy crosses the alert threshold right around the real regression point — this is why dashboards should show trend lines, not just a single aggregate number.*

## Feedback capture

Automated evals catch what you thought to test for; user feedback catches what you didn't. A minimal feedback loop just needs a way to tag a specific traced request as good/bad and to route the bad ones into your next eval-set update (closing the loop back to notebook 24's golden dataset).

In [ ]:
feedback_log = []

def capture_feedback(request_id, query, answer, rating, note=None):
    feedback_log.append({
        "request_id": request_id, "query": query, "answer": answer,
        "rating": rating, "note": note,
    })

capture_feedback("req-001", "What is a vector database?", records[0], rating="thumbs_up")
capture_feedback("req-014", "Explain KV caching", "It's when you cache the vector database.",
                  rating="thumbs_down", note="Confuses KV cache with vector DB — factual error")

thumbs_down = [f for f in feedback_log if f["rating"] == "thumbs_down"]
print(f"{len(thumbs_down)} negative-feedback item(s) — candidates to add to the golden dataset (notebook 24):")
for f in thumbs_down:
    print(f"  {f['query']!r} -> flagged: {f['note']}")


## Canary rollout: does a prompt change actually help?

Before rolling a prompt change out to 100% of traffic, run it against a slice (a "canary") and compare its eval score to the current version using the exact same harness from notebook 24 — a rollout decision backed by a measured score, not a gut feeling.

In [ ]:
CANARY_GOLDEN = [
    {"q": "What is the capital of Spain?", "ref": "Madrid"},
    {"q": "What is the capital of Egypt?", "ref": "Cairo"},
    {"q": "What is 8 + 5?", "ref": "13"},
]

CURRENT_SYSTEM = "Answer the question directly."
CANARY_SYSTEM = "Answer the question directly. Double-check arithmetic before responding."

def eval_variant(system_prompt):
    scores = []
    for item in CANARY_GOLDEN:
        out, _ = ask(item["q"], system=system_prompt, max_tokens=30, temperature=0.0)
        scores.append(exact_match_score(out, item["ref"]))
    return np.mean(scores)

current_score = eval_variant(CURRENT_SYSTEM)
canary_score = eval_variant(CANARY_SYSTEM)

print(f"Current version score: {current_score:.0%}")
print(f"Canary version score:  {canary_score:.0%}")
if canary_score > current_score:
    print("-> Canary wins: safe to roll out to more traffic.")
elif canary_score < current_score:
    print("-> Canary regresses: hold the rollout, investigate before shipping.")
else:
    print("-> No measurable difference on this slice — expand the canary sample before deciding.")


## Exercises

**Exercise 1 (Warm-up):** Change `DRIFT_POINT` and `BAD_QUALITY` in the traffic simulation and re-plot. At what quality drop does the rolling-window chart stop clearly separating from noise?

**Exercise 2 (Apply):** Implement `alert_on_drift(rolling_scores, threshold=0.8, min_consecutive=3)` that returns the index of the first point where the rolling score stays below `threshold` for at least `min_consecutive` consecutive requests (avoiding false alarms from single noisy dips).

**Exercise 3 (Extend):** Connect this notebook to notebook 33 (CI for AI): sketch how `eval_variant` above could run automatically on every prompt-change pull request, blocking the merge if the canary score regresses versus the current version.


In [ ]:
# Exercise 1: Warm-up
# Task: Try DRIFT_POINT=10, BAD_QUALITY=0.85 (a subtle regression) and re-plot rolling accuracy.
# Hint: reuse the simulation loop and the rolling-window plot cells above.

# YOUR CODE HERE


# Exercise 2: Apply
# Task: Implement alert_on_drift(rolling_scores, threshold, min_consecutive) -> int | None.
# Hint: a simple counter that resets whenever a score is >= threshold works fine.

# YOUR CODE HERE


# Exercise 3: Extend
# Task: Sketch a CI job that runs eval_variant on a prompt-change PR and blocks merge on regression.
# Hint: think about what "the canary" and "the current version" map to in a pull-request diff.

# YOUR CODE HERE


<details>
<summary>Click to reveal solutions</summary>

```python
# Exercise 1
np.random.seed(1)
subtle_scores = []
for i in range(N_REQUESTS):
    p = 0.95 if i < 10 else 0.85
    subtle_scores.append(1.0 if np.random.random() < p else 0.0)
subtle_rolling = [np.mean(subtle_scores[max(0, i - window):i + 1]) for i in range(len(subtle_scores))]
# With only a 10-point quality drop, the rolling curve is noisier and may dip below 0.8
# only briefly — this is why alert_on_drift (Exercise 2) requires several consecutive
# low points, not just one, before firing.

# Exercise 2
def alert_on_drift(rolling_scores, threshold=0.8, min_consecutive=3):
    streak = 0
    for i, s in enumerate(rolling_scores):
        streak = streak + 1 if s < threshold else 0
        if streak >= min_consecutive:
            return i - min_consecutive + 1
    return None

print(alert_on_drift(rolling))

# Exercise 3
# On every PR that touches a prompt/system-message file:
#   1. Load CURRENT_SYSTEM from main, CANARY_SYSTEM from the PR branch.
#   2. Run eval_variant(CURRENT_SYSTEM) and eval_variant(CANARY_SYSTEM) against the frozen
#      golden dataset (notebook 24) — NOT a dataset the PR author can edit in the same PR.
#   3. Post the two scores as a PR check; fail the check (block merge) if canary_score < current_score
#      by more than a small tolerance. This is exactly the regression gate notebook 33 builds.
```
</details>

## Key Takeaways
- Tracing (LangSmith `@traceable`, same pattern as notebook 21) turns "it broke" into "here's exactly what happened" — instrument production requests, not just development runs.
- Cost and latency are production metrics, not afterthoughts — track them alongside quality from the first deploy.
- A single daily-average quality number can hide a real regression; rolling-window monitoring against the notebook-24 harness catches drift a flat average misses.
- User feedback catches what your eval set didn't anticipate — route negative feedback back into the golden dataset.
- Canary rollouts turn "does this prompt change help?" into a measured score comparison instead of a guess — the same eval harness, run on a slice before full rollout.

## What's Next
Notebook 27b takes this same discipline — measurement before trust — and applies it to agents specifically: trajectory scoring, tool-call correctness, and cost-per-solved-task.
